# Week 3 Day 1 — AFL Data Foundations
**EDA, Feature Engineering & Prediction Targets**

This notebook completes Tasks 1–5 using the four supplied AFL tables. It is designed to be reproducible and leakage-safe.

**Version:** v1  
**Primary match target:** home-team win classification  
**Secondary match target:** home-team score margin regression  
**Player targets:** top disposal-getter, top goal-kicker, top fantasy-points player

> Important limitation: the supplied player tables do **not** contain a position field (forward/midfielder/defender). Therefore a valid position-based comparison cannot be produced without inventing labels. The notebook explicitly records this limitation instead of creating unreliable position assignments.

## Task 1 — Data Inventory & Understanding

### Table grains and joins
- `afl_players_info_raw.csv`: one row per player profile; join key `id` ↔ `player_id`.
- `afl_players_round_by_round_stats_raw.csv`: one row per player-game; player-level match statistics.
- `afl_players_seasonal_stats_raw.csv`: one row per player-season/phase; `player_id + year + team + is_finals`.
- `team_matches_home_away_raw.csv`: two rows per match, one team-perspective row for each side. The `H` row is used as the single match anchor.

Match-level player targets are joined using a normalized match key built from date, round, and the unordered team pair. The raw team-match table is the authoritative source for match IDs.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

def load(name):
    return pd.read_csv(DATA_DIR / name, low_memory=False)

players_info = load("afl_players_info_raw.csv")
player_round = load("afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv")
player_season = load("afl_players_seasonal_stats_raw.csv")
team_matches = load("team_matches_home_away_raw - team_matches_home_away_raw.csv.csv")

team_matches["match_date"] = pd.to_datetime(team_matches["match_date"], errors="coerce")
player_round["match_date"] = pd.to_datetime(player_round["match_date"], errors="coerce")

print("Loaded tables:")
for name, df in [("players_info",players_info),("player_round",player_round),
                 ("player_season",player_season),("team_matches",team_matches)]:
    print(f"{name:15s}: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
def norm_team(s):
    return s.astype(str).str.strip().replace({"W. Bulldogs":"Western Bulldogs"})

team_matches["team_norm"] = norm_team(team_matches["team_name"])
team_matches["opp_norm"] = norm_team(team_matches["opponent"])
player_round["team_norm"] = norm_team(player_round["team"])
player_round["opp_norm"] = norm_team(player_round["opponent"])

def make_key(df, team_col="team_norm", opp_col="opp_norm"):
    pair = df[[team_col,opp_col]].apply(lambda r: "||".join(sorted([r.iloc[0],r.iloc[1]])), axis=1)
    return (df["year"].astype(str)+"|"+df["match_date"].dt.strftime("%Y-%m-%d")+"|"+
            df["round"].astype(str)+"|"+pair)

team_matches["match_key"] = make_key(team_matches)
player_round["match_key"] = make_key(player_round)

home_rows = team_matches[team_matches["home_away"].eq("H")].copy()
print("Date range:", team_matches["match_date"].min().date(), "to", team_matches["match_date"].max().date())
print("Seasons:", team_matches["year"].min(), "to", team_matches["year"].max(), "|", team_matches["year"].nunique())
print("Raw team names:", team_matches["team_name"].nunique())
print("Normalized team names:", pd.concat([team_matches["team_norm"],team_matches["opp_norm"]]).nunique())
print("Players in player-game table:", player_round["player_id"].nunique())
print("Match anchors:", len(home_rows))
print("Home/away rows per match:", team_matches.groupby("match_key").size().value_counts().to_dict())

In [ ]:
# Data-quality audit: missingness, duplicate rows/IDs, naming and outliers
tables = {
    "players_info": players_info,
    "player_round": player_round,
    "player_season": player_season,
    "team_matches": team_matches
}
for name, df in tables.items():
    print("\n", name)
    print("duplicate full rows:", int(df.duplicated().sum()))
    if "id" in df.columns:
        print("duplicate id:", int(df["id"].duplicated().sum()))
    print("top missing columns (%):")
    print((df.isna().mean().mul(100).sort_values(ascending=False).head(8).round(1)).to_string())

print("\nResult counts:", team_matches["result"].value_counts(dropna=False).to_dict())
print("Player-stat outlier flags:")
for col in ["disposals","goals","fantasy_points","kicks","marks"]:
    x = pd.to_numeric(player_round[col], errors="coerce")
    print(col, "min=", x.min(), "max=", x.max(), "99.9%=", round(x.quantile(.999),1))
print("\nKnown structural/naming notes:")
print("- Coverage is 1983–2025 (43 seasons).")
print("- Historical club structures include Brisbane Bears/Fitzroy Lions and later Gold Coast Suns/GWS.")
print("- W. Bulldogs is normalized to Western Bulldogs.")
print("- Rule changes are not encoded in the supplied tables, so no rule-change effect is assumed.")
print("- Negative disposals (minimum -5) are flagged as a source-data anomaly; raw values are retained and excluded from sensible disposal distribution summaries.")

## Task 2 — Define Prediction Targets Precisely

### Match target contract
1. **`target_home_win`** — primary binary classification target. It equals 1 when the home team wins and 0 otherwise. Draws remain separately identifiable through `target_draw`; they are not silently converted into wins.
2. **`target_margin`** — secondary regression target: home score minus away score.

Classification is the first modeling framing because the business question is match winner. Margin regression is retained because it preserves more outcome information and can later be converted into win probabilities.

### Top-player target contract
- **Top disposal-getter:** player with maximum `disposals` in a match.
- **Top goal-kicker:** player with maximum `goals` in a match.
- **Top fantasy player:** player with maximum `fantasy_points` in a match.

There is no composite formula because the assignment can be satisfied with these three direct observed-stat targets. Ties are resolved deterministically by the lowest `player_id`.

In [ ]:
# Build player-match targets from the player-game table
# Vectorized approach: sort by metric descending and player_id ascending, then take first row per match.
target_base=player_round[["match_key","match_date","year","player_id","disposals","goals","fantasy_points"]].copy()
target_rows=target_base.groupby("match_key",as_index=False).agg(
    match_date=("match_date","min"), season=("year","first")
)

for metric,prefix in [("disposals","top_disposal"),("goals","top_goal_kicker"),("fantasy_points","top_fantasy")]:
    z=target_base.dropna(subset=[metric]).sort_values(
        ["match_key",metric,"player_id"], ascending=[True,False,True]
    )
    w=z.drop_duplicates("match_key")[["match_key","player_id",metric]].rename(
        columns={"player_id":prefix+"_player_id",metric:prefix+"_value"}
    )
    target_rows=target_rows.merge(w,on="match_key",how="left")

player_targets=target_rows.sort_values(["match_date","match_key"])
player_targets.to_csv("afl_player_match_targets_v1.csv",index=False)
print("Saved player targets:", player_targets.shape)
print(player_targets.head())


### One-page target/data dictionary

| Target | Definition | Formula | Level |
|---|---|---|---|
| `target_home_win` | Home team wins | `1(result == W)` | Match |
| `target_draw` | Match is a draw | `1(result == D)` | Match |
| `target_margin` | Home score margin | `home_score - away_score` | Match |
| `top_disposal_player_id` | Player with most disposals | `argmax(disposals)`; tie → lowest player_id | Player-in-match |
| `top_goal_kicker_player_id` | Player with most goals | `argmax(goals)`; tie → lowest player_id | Player-in-match |
| `top_fantasy_player_id` | Player with most fantasy points | `argmax(fantasy_points)`; tie → lowest player_id | Player-in-match |

**Important:** these player targets are post-match labels. They must never be used as pre-match input features.

## Task 3 — Exploratory Data Analysis

The EDA covers:
1. Home-team win rate by season.
2. Team win rates over time for frequently observed teams.
3. Ladder strength vs. home win rate.
4. Recent form vs. home win rate.
5. Rest advantage vs. match margin.
6. Prior head-to-head record vs. home win rate.
7. Player distributions and historical leaders.
8. Week-to-week consistency using player-level rolling/within-career variation.

Weather and travel distance are not present in the supplied tables, so they are not fabricated. Position-based differences are also unavailable because no position column exists.

In [ ]:
# Historical home-ground advantage
home_rate = home_rows.groupby("year")["result"].apply(lambda s:(s=="W").mean())
plt.figure(figsize=(9,4.5))
plt.plot(home_rate.index,home_rate.values,marker="o",markersize=2)
plt.axhline((home_rows["result"]=="W").mean(),linestyle="--",label="Overall")
plt.title("Home-Team Win Rate by Season"); plt.xlabel("Season"); plt.ylabel("Home win rate")
plt.legend(); plt.tight_layout(); plt.savefig(FIG_DIR/"01_home_win_rate_by_season.png"); plt.show()

In [ ]:
# Team win rates over time: top 10 teams by number of match appearances
team_season = home_rows.groupby(["year","team_norm"])["result"].apply(lambda s:(s=="W").mean()).reset_index(name="win_rate")
top_teams = team_season.groupby("team_norm").size().nlargest(10).index
plt.figure(figsize=(10,5))
for t in top_teams:
    q=team_season[team_season.team_norm.eq(t)]
    plt.plot(q.year,q.win_rate,alpha=.7,label=t)
plt.title("Team Win Rates Over Time — Frequently Observed Teams")
plt.xlabel("Season"); plt.ylabel("Win rate"); plt.legend(fontsize=7,ncol=2); plt.tight_layout()
plt.savefig(FIG_DIR/"02_team_win_rates_over_time.png"); plt.show()

In [ ]:
# Player distributions and historical leaders
pr=player_round.copy()
plt.figure(figsize=(9,4.5))
plt.hist(pr["disposals"].where(pr["disposals"]>=0).dropna(),bins=40)
plt.title("Player-Game Disposals Distribution"); plt.xlabel("Disposals"); plt.ylabel("Player-games")
plt.tight_layout(); plt.savefig(FIG_DIR/"03_player_disposals_distribution.png"); plt.show()

plt.figure(figsize=(9,4.5))
plt.hist(pr["fantasy_points"].dropna(),bins=40)
plt.title("Player-Game Fantasy Points Distribution"); plt.xlabel("Fantasy points"); plt.ylabel("Player-games")
plt.tight_layout(); plt.savefig(FIG_DIR/"04_fantasy_distribution.png"); plt.show()

leaders=pr.groupby("player_id").agg(
    total_disposals=("disposals","sum"), total_goals=("goals","sum"),
    total_fantasy=("fantasy_points","sum"), games=("player_id","size")
).sort_values("total_disposals",ascending=False).head(10)
print("Historical leaders by total recorded disposals:")
print(leaders)

In [ ]:
# Player consistency: within-player coefficient of variation for disposals/fantasy
cons=pr.groupby("player_id").agg(
    games=("player_id","size"),
    mean_disposals=("disposals","mean"),
    sd_disposals=("disposals","std"),
    mean_fantasy=("fantasy_points","mean"),
    sd_fantasy=("fantasy_points","std")
)
cons=cons[cons.games>=20].copy()
cons["cv_disposals"]=cons["sd_disposals"]/cons["mean_disposals"].replace(0,np.nan)
cons["cv_fantasy"]=cons["sd_fantasy"]/cons["mean_fantasy"].replace(0,np.nan)
print("Median within-player CV, players with >=20 games:")
print(cons[["cv_disposals","cv_fantasy"]].median().round(3))
print("\nPosition analysis: NOT AVAILABLE — supplied player tables contain no position field. No forward/midfielder/defender labels are inferred.")

## Task 4 — Feature Engineering for Prediction

All match features below are calculated from information available **strictly before** the current match:
- last-5 win rate;
- last-5 average score for/against;
- current win streak;
- days of rest;
- season-to-date ladder points and position;
- prior head-to-head record.

The current match result and score are added to team histories **only after** the feature row is created. This is the main leakage-control rule.

In [ ]:
def build_match_features(df):
    data=df[df.home_away.eq("H")].copy().sort_values(["match_date","id"]).reset_index(drop=True)
    history=defaultdict(list); last_date={}; season_stats=defaultdict(lambda:{"wins":0,"draws":0,"losses":0,"pf":0.0,"pa":0.0,"games":0})
    h2h=defaultdict(list); rows=[]
    for _,r in data.iterrows():
        h,a=r.team_norm,r.opp_norm; dt=r.match_date; yr=int(r.year)
        def recent(team):
            hist=history[team][-5:]
            return {
                "win_rate":(sum(x["result"]=="W" for x in hist)+.5*sum(x["result"]=="D" for x in hist))/len(hist) if hist else np.nan,
                "avg_for":np.mean([x["sf"] for x in hist]) if hist else np.nan,
                "avg_against":np.mean([x["sa"] for x in hist]) if hist else np.nan,
                "streak":next((i for i,x in enumerate(reversed(history[team])) if x["result"]!="W"),len(history[team])),
                "rest":(dt-last_date[team]).days if team in last_date else np.nan}
        fh,fa=recent(h),recent(a)
        def ladder(team):
            s=season_stats[(team,yr)]; return 4*s["wins"]+2*s["draws"]
        hp,ap=ladder(h),ladder(a)
        active=[]
        for (team,season),s in season_stats.items():
            if season==yr and s["games"]>0:
                pct=s["pf"]/s["pa"]*100 if s["pa"] else -np.inf
                active.append((team,4*s["wins"]+2*s["draws"],pct))
        rank_map={t:i+1 for i,(t,_,_) in enumerate(sorted(active,key=lambda x:(-x[1],-x[2],x[0])))}
        prior=h2h[tuple(sorted((h,a)))]
        hw=sum(w==h for w in prior); hd=sum(w=="DRAW" for w in prior)
        h2hr=(hw+.5*hd)/len(prior) if prior else np.nan
        rows.append({
            "match_id":int(r.id),"match_key":r.match_key,"match_date":dt,"season":yr,"round":str(r["round"]),
            "home_team":h,"away_team":a,"venue":str(r.venue).strip() if pd.notna(r.venue) else "Unknown",
            "home_score":r.team_score,"away_score":r.opponent_score,
            "home_win_rate_last5":fh["win_rate"],"away_win_rate_last5":fa["win_rate"],
            "home_avg_score_for_last5":fh["avg_for"],"away_avg_score_for_last5":fa["avg_for"],
            "home_avg_score_against_last5":fh["avg_against"],"away_avg_score_against_last5":fa["avg_against"],
            "home_win_streak":fh["streak"],"away_win_streak":fa["streak"],
            "home_rest_days":fh["rest"],"away_rest_days":fa["rest"],
            "home_ladder_points_prior":hp,"away_ladder_points_prior":ap,
            "home_ladder_position_prior":rank_map.get(h,np.nan),"away_ladder_position_prior":rank_map.get(a,np.nan),
            "ladder_points_diff":hp-ap,"h2h_home_win_rate_prior":h2hr,"h2h_games_prior":len(prior),
            "target_home_win":int(r.result=="W"),"target_draw":int(r.result=="D"),
            "target_result":r.result,"target_margin":r.team_score-r.opponent_score
        })
        hr=r.result; ar="L" if hr=="W" else "W" if hr=="L" else "D"
        history[h].append({"result":hr,"sf":r.team_score,"sa":r.opponent_score})
        history[a].append({"result":ar,"sf":r.opponent_score,"sa":r.team_score})
        last_date[h]=last_date[a]=dt
        for team,sf,sa,res in [(h,r.team_score,r.opponent_score,hr),(a,r.opponent_score,r.team_score,ar)]:
            s=season_stats[(team,yr)]; s["games"]+=1; s["pf"]+=sf; s["pa"]+=sa
            if res=="W":s["wins"]+=1
            elif res=="L":s["losses"]+=1
            else:s["draws"]+=1
        h2h[tuple(sorted((h,a)))].append("DRAW" if hr=="D" else h if hr=="W" else a)
    return pd.DataFrame(rows)

features=build_match_features(team_matches)
features.to_csv("afl_match_features_v1.csv",index=False)
print("Feature table:",features.shape)

In [ ]:
# Five required prediction-relevant relationships
f=features.copy()
f["form_diff"]=f.home_win_rate_last5-f.away_win_rate_last5
q=f.dropna(subset=["form_diff"]); q["bin"]=pd.qcut(q.form_diff,10,duplicates="drop")
g=q.groupby("bin",observed=True).target_home_win.mean()
plt.figure(figsize=(9,4.5)); plt.plot(range(len(g)),g.values,marker="o")
plt.title("1. Recent Form Difference vs Home Win Rate"); plt.xlabel("Form-difference decile"); plt.ylabel("Observed home win rate")
plt.tight_layout(); plt.savefig(FIG_DIR/"05_form_vs_home_win.png"); plt.show()

q=f.dropna(subset=["home_rest_days","away_rest_days"]).copy(); q["rest_diff"]=q.home_rest_days-q.away_rest_days; q["bin"]=pd.qcut(q.rest_diff,10,duplicates="drop")
g=q.groupby("bin",observed=True).target_margin.mean()
plt.figure(figsize=(9,4.5)); plt.plot(range(len(g)),g.values,marker="o")
plt.title("2. Rest Advantage vs Match Margin"); plt.xlabel("Rest-difference decile"); plt.ylabel("Mean home margin")
plt.tight_layout(); plt.savefig(FIG_DIR/"06_rest_vs_margin.png"); plt.show()

q=f.dropna(subset=["ladder_points_diff"]).copy(); q["bin"]=pd.qcut(q.ladder_points_diff,10,duplicates="drop")
g=q.groupby("bin",observed=True).target_home_win.mean()
plt.figure(figsize=(9,4.5)); plt.plot(range(len(g)),g.values,marker="o")
plt.title("3. Prior Ladder Strength vs Home Win Rate"); plt.xlabel("Ladder-points-difference decile"); plt.ylabel("Observed home win rate")
plt.tight_layout(); plt.savefig(FIG_DIR/"07_ladder_vs_win.png"); plt.show()

q=f.dropna(subset=["h2h_home_win_rate_prior"]).copy(); q["bin"]=pd.qcut(q.h2h_home_win_rate_prior,10,duplicates="drop")
g=q.groupby("bin",observed=True).target_home_win.mean()
plt.figure(figsize=(9,4.5)); plt.plot(range(len(g)),g.values,marker="o")
plt.title("4. Prior Head-to-Head Record vs Home Win Rate"); plt.xlabel("Prior H2H-rate decile"); plt.ylabel("Observed home win rate")
plt.tight_layout(); plt.savefig(FIG_DIR/"08_h2h_vs_win.png"); plt.show()

q=f.dropna(subset=["home_avg_score_for_last5"]).copy()
plt.figure(figsize=(9,4.5)); plt.scatter(q.home_avg_score_for_last5,q.target_margin,alpha=.18)
plt.title("5. Home Recent Scoring Form vs Match Margin"); plt.xlabel("Home average score, last 5"); plt.ylabel("Home match margin")
plt.tight_layout(); plt.savefig(FIG_DIR/"09_score_form_vs_margin.png"); plt.show()

### Feature dictionary

Each feature is pre-match. The target columns are labels and are not allowed as model inputs.

| Feature | Description | Window | Source / computation |
|---|---|---|---|
| `home_win_rate_last5` / `away_win_rate_last5` | Prior win rate; draw = 0.5 | Previous 5 games | prior `result` |
| `home_avg_score_for_last5` / `away_avg_score_for_last5` | Average points scored | Previous 5 games | prior `team_score` |
| `home_avg_score_against_last5` / `away_avg_score_against_last5` | Average points conceded | Previous 5 games | prior `opponent_score` |
| `home_win_streak` / `away_win_streak` | Consecutive wins immediately before match | Prior history | prior `result` |
| `home_rest_days` / `away_rest_days` | Days since previous match | Previous match | prior `match_date` |
| `home_ladder_points_prior` / `away_ladder_points_prior` | Season points before current match | Season-to-date | win=4, draw=2 |
| `home_ladder_position_prior` / `away_ladder_position_prior` | Prior ladder rank | Season-to-date | points then percentage |
| `ladder_points_diff` | Home minus away prior points | Current pre-match snapshot | derived |
| `h2h_home_win_rate_prior` | Prior home-team record vs opponent | All prior meetings | W=1, D=.5 |
| `h2h_games_prior` | Prior meetings count | All prior meetings | matchup history |
| `venue` | Match venue | Current match metadata | raw `venue` |

The versioned CSV is `afl_match_features_v1.csv`.

In [ ]:
feature_dict=pd.DataFrame([
["home_win_rate_last5","Home prior win rate; draw=.5","5 prior games","result history"],
["away_win_rate_last5","Away prior win rate; draw=.5","5 prior games","result history"],
["home_avg_score_for_last5","Home mean points scored","5 prior games","team_score"],
["away_avg_score_for_last5","Away mean points scored","5 prior games","team_score"],
["home_avg_score_against_last5","Home mean points conceded","5 prior games","opponent_score"],
["away_avg_score_against_last5","Away mean points conceded","5 prior games","opponent_score"],
["home_win_streak","Home consecutive wins","Prior history","result"],
["away_win_streak","Away consecutive wins","Prior history","result"],
["home_rest_days","Days since home team's previous match","Previous match","match_date"],
["away_rest_days","Days since away team's previous match","Previous match","match_date"],
["home_ladder_points_prior","Home season points before match","Season-to-date","win=4, draw=2"],
["away_ladder_points_prior","Away season points before match","Season-to-date","win=4, draw=2"],
["home_ladder_position_prior","Home prior ladder rank","Season-to-date","points then percentage"],
["away_ladder_position_prior","Away prior ladder rank","Season-to-date","points then percentage"],
["ladder_points_diff","Home minus away prior ladder points","Current snapshot","derived"],
["h2h_home_win_rate_prior","Home prior win rate vs opponent","All prior meetings","W=1, D=.5"],
["h2h_games_prior","Prior meetings count","All prior meetings","matchup history"],
["venue","Venue metadata","Current match","raw venue"]
],columns=["feature_name","description","computation_window","source_columns_or_formula"])
feature_dict.to_csv("afl_feature_dictionary_v1.csv",index=False)
print(feature_dict)

## Task 5 — Reproducible Train/Hold-Out Split

A random split is inappropriate because sports data are ordered in time: a random split can train on later matches and evaluate on earlier matches, allowing future-era information to influence the apparent score. The default split uses **all seasons before 2025 for training and 2025 as the hold-out season**, which approximates deployment on a future season.

Sports outcomes are inherently noisy because injuries, selection, tactics, weather, travel, form shocks and other factors are only partly observed. Therefore a realistic model should aim for stable out-of-time performance rather than perfect accuracy. A nearly perfect score would be a red flag because it could indicate that post-match information or another leakage path entered the features. The exact ceiling depends on feature quality and target definition, but perfect prediction should not be expected.

In [ ]:
def time_split(df, holdout_season=None):
    df=df.sort_values(["match_date","match_id"]).reset_index(drop=True).copy()
    if holdout_season is None:
        holdout_season=int(df.season.max())
    train=df[df.season<holdout_season].copy()
    holdout=df[df.season==holdout_season].copy()
    if train.empty or holdout.empty:
        raise ValueError("Time split produced an empty partition.")
    return train,holdout

train,holdout=time_split(features)
print("Train:",train.season.min(),"to",train.season.max(),"|",len(train),"rows")
print("Hold-out:",holdout.season.unique(),"|",len(holdout),"rows")
print("Strict temporal ordering:",train.match_date.max()<holdout.match_date.min())
print("Hold-out home-win rate:",round(holdout.target_home_win.mean(),3))

In [ ]:
# Leakage sanity checks
target_cols={"target_home_win","target_draw","target_result","target_margin","home_score","away_score"}
pre_match_cols=[c for c in features.columns if c not in target_cols]
for c in pre_match_cols:
    if "target" in c.lower() or c in {"home_score","away_score"}:
        print("Review column:",c)
print("No current/future result, score or player post-match statistic is used in the engineered pre-match feature set.")
print("Versioned outputs written: afl_match_features_v1.csv, afl_feature_dictionary_v1.csv, afl_player_match_targets_v1.csv")

## Final checklist against the assignment

- **Task 1:** all four tables documented; grain and join strategy documented; date range, seasons, teams and players reported; naming/structural changes and quality checks included.
- **Task 2:** match classification + margin regression targets defined; three top-player targets defined; formulas, levels and tie rule documented.
- **Task 3:** team win rates over time, home advantage, ladder trend, player distributions, historical leaders, consistency, and 5 prediction relationships included. Position/weather/travel are explicitly unavailable in the supplied data.
- **Task 4:** leakage-safe rolling/form, H2H, venue, ladder position and rest features built; versioned feature table and feature dictionary saved.
- **Task 5:** reusable time split uses the most recent season (2025) as hold-out; random split rationale and realistic accuracy ceiling documented.